In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("../data/raw/predictive_maintenance_v3.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# on garde machine_id et timestamp pour le calcul des deltas
df_model = df.drop(columns=["estimated_repair_cost"], errors="ignore").copy()

print(df_model.shape)
print(df_model.columns.tolist())

(24042, 14)
['timestamp', 'machine_id', 'machine_type', 'vibration_rms', 'temperature_motor', 'current_phase_avg', 'pressure_level', 'rpm', 'operating_mode', 'hours_since_maintenance', 'ambient_temp', 'rul_hours', 'failure_within_24h', 'failure_type']


In [2]:
base_cols = ["vibration_rms", "temperature_motor", "pressure_level", "rpm"]

print(df_model[base_cols].isna().sum())
print(df_model[base_cols].dtypes)
print(df_model[base_cols].head())

vibration_rms        1000
temperature_motor     834
pressure_level        924
rpm                   533
dtype: int64
vibration_rms        float64
temperature_motor    float64
pressure_level       float64
rpm                  float64
dtype: object
   vibration_rms  temperature_motor  pressure_level    rpm
0           0.81              49.51            23.6  860.9
1           0.75              40.58            23.6  899.6
2           0.71              49.70            21.3  862.7
3           0.76              43.04            22.6  870.4
4           0.88              41.39            22.2  881.9


In [3]:
df_model = df_model.sort_values(["machine_id", "timestamp"]).reset_index(drop=True)

df_model["vibration_delta"] = df_model.groupby("machine_id")["vibration_rms"].diff()
df_model["temperature_delta"] = df_model.groupby("machine_id")["temperature_motor"].diff()

print(df_model[["vibration_delta", "temperature_delta"]].isna().sum())
print(df_model[["vibration_delta", "temperature_delta"]].head(10))

vibration_delta      1980
temperature_delta    1663
dtype: int64
   vibration_delta  temperature_delta
0              NaN                NaN
1            -0.06              -8.93
2            -0.04               9.12
3             0.05              -6.66
4             0.12              -1.65
5             0.06               0.61
6            -0.06              -0.29
7             2.05              12.86
8            -2.19              -9.62
9             0.16              -1.83


In [4]:
df_model["vibration_delta"] = df_model["vibration_delta"].fillna(0)
df_model["temperature_delta"] = df_model["temperature_delta"].fillna(0)

df_model["anomaly_trend_raw"] = (
    0.6 * df_model["vibration_delta"].clip(lower=0) +
    0.4 * df_model["temperature_delta"].clip(lower=0)
)

print(df_model[["anomaly_trend_raw"]].isna().sum())
print(df_model[["anomaly_trend_raw"]].describe())

anomaly_trend_raw    0
dtype: int64
       anomaly_trend_raw
count       24042.000000
mean            1.719876
std             2.793895
min             0.000000
25%             0.000000
50%             0.156000
75%             2.428000
max            20.372000


In [5]:
score_cols = [
    "vibration_rms",
    "temperature_motor",
    "pressure_level",
    "rpm",
    "anomaly_trend_raw"
]

print(df_model[score_cols].isna().sum())
print(df_model[score_cols].dtypes)

vibration_rms        1000
temperature_motor     834
pressure_level        924
rpm                   533
anomaly_trend_raw       0
dtype: int64
vibration_rms        float64
temperature_motor    float64
pressure_level       float64
rpm                  float64
anomaly_trend_raw    float64
dtype: object


In [6]:
for col in score_cols:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")
    df_model[col] = df_model[col].fillna(df_model[col].median())

print(df_model[score_cols].isna().sum())
print(df_model[score_cols].describe())

vibration_rms        0
temperature_motor    0
pressure_level       0
rpm                  0
anomaly_trend_raw    0
dtype: int64
       vibration_rms  temperature_motor  pressure_level           rpm  \
count   24042.000000       24042.000000    24042.000000  24042.000000   
mean        1.608956          51.357662       58.523667   1138.445662   
std         1.060691          12.302671       38.050389    903.498629   
min         0.350000          28.000000       10.100000    124.100000   
25%         0.850000          42.890000       23.100000    496.025000   
50%         1.270000          50.060000       46.300000    856.000000   
75%         2.230000          59.600000       90.800000   1667.875000   
max        10.000000          95.000000      206.500000   4098.800000   

       anomaly_trend_raw  
count       24042.000000  
mean            1.719876  
std             2.793895  
min             0.000000  
25%             0.000000  
50%             0.156000  
75%             2.428000 

In [7]:
scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(df_model[score_cols])

scaled_df = pd.DataFrame(
    scaled_values,
    columns=[f"{col}_norm" for col in score_cols],
    index=df_model.index
)

print(scaled_df.isna().sum())
print(scaled_df.head())
print(scaled_df.describe())

vibration_rms_norm        0
temperature_motor_norm    0
pressure_level_norm       0
rpm_norm                  0
anomaly_trend_raw_norm    0
dtype: int64
   vibration_rms_norm  temperature_motor_norm  pressure_level_norm  rpm_norm  \
0            0.047668                0.321045             0.068737  0.185372   
1            0.041451                0.187761             0.068737  0.195109   
2            0.037306                0.323881             0.057026  0.185825   
3            0.042487                0.224478             0.063646  0.187763   
4            0.054922                0.199851             0.061609  0.190656   

   anomaly_trend_raw_norm  
0                0.000000  
1                0.000000  
2                0.179069  
3                0.001473  
4                0.003534  
       vibration_rms_norm  temperature_motor_norm  pressure_level_norm  \
count        24042.000000            24042.000000         24042.000000   
mean             0.130462                0.348622 

In [8]:
df_model = pd.concat([df_model, scaled_df], axis=1)

df_model["health_score"] = 1 - (
    0.30 * df_model["vibration_rms_norm"] +
    0.25 * df_model["temperature_motor_norm"] +
    0.20 * df_model["pressure_level_norm"] +
    0.15 * df_model["rpm_norm"] +
    0.10 * df_model["anomaly_trend_raw_norm"]
)

df_model["health_score"] = df_model["health_score"].clip(0, 1)

print(df_model["health_score"].isna().sum())
print(df_model["health_score"].describe())
print(df_model[["health_score"]].head())

0
count    24042.000000
mean         0.777672
std          0.139455
min          0.221575
25%          0.661253
50%          0.829878
75%          0.886833
max          0.992878
Name: health_score, dtype: float64
   health_score
0      0.863885
1      0.897611
2      0.850652
3      0.890094
4      0.892287


In [9]:
conditions = [
    df_model["health_score"] >= 0.8,
    (df_model["health_score"] >= 0.5) & (df_model["health_score"] < 0.8),
    df_model["health_score"] < 0.5
]
labels = ["good", "warning", "critical"]

df_model["health_status"] = np.select(conditions, labels, default="unknown")

print(df_model["health_status"].value_counts(dropna=False))
df_model[["health_score", "health_status"]].head()

health_status
good        13159
warning     10060
critical      823
Name: count, dtype: int64


,health_score,health_status
0,0.863885,good
1,0.897611,good
2,0.850652,good
3,0.890094,good
4,0.892287,good
